# BCI Project: Real-Time Two-Stage Decoder (Safety-First)

Priority order implemented:
1. Minimize false triggers (especially wrong/false commands)
2. Lower latency
3. Improve Left/Right accuracy

Design constraints:
- `Repeated` is `Active` for intent-gate training only; not a command class.
- Strict no-overlap windows (`1.0s` window, `1.0s` hop).
- Trial-level grouped validation.
- Decoder logic uses hysteresis + debounce + cooldown + direction confidence gating.

In [ ]:
import numpy as np
import pandas as pd

from bci_pipeline import (
    DecoderConfig,
    load_openbci_tsv,
    channel_quality_report,
    build_trial_segments,
    build_feature_dataset,
    build_model_bank,
    evaluate_models_grouped,
    split_grouped_holdout_stratified,
    train_two_stage_models,
    run_holdout_simulation,
    evaluate_two_stage_over_seeds,
    tune_decoder_controls,
    conservative_decoder_config,
)

# Update paths if needed
LR_FILE = '/Users/yanivnaggar/Desktop/Spring 2026/IS/OPENBCI_runs/Yaniv/EEG_LR/LR-2-27-26-(01).csv'
HR_FILE = '/Users/yanivnaggar/Desktop/Spring 2026/IS/OPENBCI_runs/Yaniv/EMG_JvsN/HR-2-27-26-(01).csv'

FS = 250.0
WINDOW_SEC = 1.0
HOP_SEC = 1.0  # strict no-overlap
BASELINE_BUFFER_SAMPLES = 500
RANDOM_STATE = 42

## 1) Load and Inspect

In [ ]:
lr_df = load_openbci_tsv(LR_FILE)
hr_df = load_openbci_tsv(HR_FILE)

print('LR shape:', lr_df.shape)
print('HR shape:', hr_df.shape)
print('\nLR channel quality:')
display(channel_quality_report(lr_df))
print('\nHR channel quality:')
display(channel_quality_report(hr_df))

## 2) Segment Trials and Baseline

In [ ]:
trials = build_trial_segments(
    lr_df=lr_df,
    hr_df=hr_df,
    baseline_buffer_samples=BASELINE_BUFFER_SAMPLES,
)

for label in ['Norm', 'Left', 'Right', 'Repeated']:
    print(f'{label}: {len(trials[label])} trial segments')

## 3) Build Feature Dataset

In [ ]:
df_feat = build_feature_dataset(
    trials_by_class=trials,
    fs=FS,
    window_sec=WINDOW_SEC,
    hop_sec=HOP_SEC,
    include_asymmetry=True,
)

meta_cols = ['Label', 'IntentLabel', 'GroupID', 'TrialIndexWithinClass']
feature_cols = [c for c in df_feat.columns if c not in meta_cols]

X = df_feat[feature_cols].to_numpy()
y_label = df_feat['Label'].to_numpy()
y_intent = df_feat['IntentLabel'].to_numpy()
groups = df_feat['GroupID'].to_numpy()

mask_lr = np.isin(y_label, ['Left', 'Right'])
X_lr = X[mask_lr]
y_lr = y_label[mask_lr]
groups_lr = groups[mask_lr]

print('Feature dataset shape:', df_feat.shape)
print('Label counts:')
display(df_feat['Label'].value_counts())
print('Intent counts:')
display(df_feat['IntentLabel'].value_counts())
print('Unique groups:', len(np.unique(groups)))

## 4) Grouped CV Model Selection

In [ ]:
models = build_model_bank(random_state=RANDOM_STATE)

intent_cv = evaluate_models_grouped(X, y_intent, groups, models, n_splits=5)
direction_cv = evaluate_models_grouped(X_lr, y_lr, groups_lr, models, n_splits=5)

print('Intent CV (Norm vs Active):')
display(intent_cv)
print('Direction CV (Left vs Right):')
display(direction_cv)

intent_model_name = intent_cv.iloc[0]['Model']
direction_model_name = direction_cv.iloc[0]['Model']
intent_model = models[intent_model_name]
direction_model = models[direction_model_name]

print('Selected intent model:', intent_model_name)
print('Selected direction model:', direction_model_name)

## 5) Logic-First Control Tuning (Safety-First Ranking)

In [ ]:
seeds = [11, 23, 37, 42, 71, 89]

candidate_configs = [
    DecoderConfig(0.70, 0.55, 3, 4, 3, 2, 0.60, 0.10),
    DecoderConfig(0.75, 0.60, 3, 4, 3, 2, 0.65, 0.12),
    DecoderConfig(0.80, 0.65, 3, 5, 3, 2, 0.70, 0.15),
    DecoderConfig(0.75, 0.60, 4, 5, 3, 2, 0.65, 0.12),
    DecoderConfig(0.80, 0.65, 4, 6, 3, 2, 0.70, 0.15),
]

tuning_df = tune_decoder_controls(
    X=X,
    y_intent=y_intent,
    y_label=y_label,
    groups=groups,
    intent_model=intent_model,
    direction_model=direction_model,
    configs=candidate_configs,
    seeds=seeds,
    test_size=0.25,
    window_sec=WINDOW_SEC,
    hop_sec=HOP_SEC,
    stratified_group_split=True,
)

print('Safety-first tuning results (top = safest):')
display(tuning_df)

best = tuning_df.iloc[0]
best_cfg = DecoderConfig(
    enter_active_threshold=float(best['enter_active_threshold']),
    exit_active_threshold=float(best['exit_active_threshold']),
    k_consecutive=int(best['k_consecutive']),
    cooldown_windows=int(best['cooldown_windows']),
    majority_windows=int(best['majority_windows']),
    min_direction_votes=int(best['min_direction_votes']),
    direction_min_confidence=float(best['direction_min_confidence']),
    direction_margin=float(best['direction_margin']),
)
print('Chosen decoder config:')
print(best_cfg)

## 6) Multi-Seed Holdout Evaluation with Chosen Controls

In [ ]:
seed_eval_df, seed_summary = evaluate_two_stage_over_seeds(
    X=X,
    y_intent=y_intent,
    y_label=y_label,
    groups=groups,
    intent_model=intent_model,
    direction_model=direction_model,
    seeds=seeds,
    test_size=0.25,
    window_sec=WINDOW_SEC,
    hop_sec=HOP_SEC,
    decoder_config=best_cfg,
    stratified_group_split=True,
)

print('Per-seed holdout results:')
display(seed_eval_df)
print('Aggregate summary:')
for k, v in seed_summary.items():
    print(f'{k}: {v}')

## 7) Single Reference Split (Detailed Output)

In [ ]:
train_idx, test_idx = split_grouped_holdout_stratified(
    groups=groups,
    y_group_label=y_label,
    test_size=0.25,
    random_state=RANDOM_STATE,
)

X_train, X_test = X[train_idx], X[test_idx]
y_train_intent = y_intent[train_idx]
y_train_label, y_test_label = y_label[train_idx], y_label[test_idx]
groups_test = groups[test_idx]

intent_pipe, direction_pipe = train_two_stage_models(
    X_train=X_train,
    y_train_intent=y_train_intent,
    y_train_label=y_train_label,
    intent_model=intent_model,
    direction_model=direction_model,
)

sim = run_holdout_simulation(
    X_test=X_test,
    y_test_label=y_test_label,
    groups_test=groups_test,
    intent_model=intent_pipe,
    direction_model=direction_pipe,
    window_sec=WINDOW_SEC,
    hop_sec=HOP_SEC,
    enter_active_threshold=best_cfg.enter_active_threshold,
    exit_active_threshold=best_cfg.exit_active_threshold,
    k_consecutive=best_cfg.k_consecutive,
    cooldown_windows=best_cfg.cooldown_windows,
    majority_windows=best_cfg.majority_windows,
    min_direction_votes=best_cfg.min_direction_votes,
    direction_min_confidence=best_cfg.direction_min_confidence,
    direction_margin=best_cfg.direction_margin,
)

print('Direction trials (Left/Right):', sim['total_direction_trials'])
print('Correct:', sim['correct'])
print('Wrong:', sim['wrong'])
print('Missed:', sim['missed'])
print(f"Direction accuracy: {sim['accuracy_on_direction_trials']:.3f}")
print(f"Norm false triggers/min: {sim['false_triggers_per_min']:.3f}")
print(f"Repeated off-target triggers/min: {sim['repeated_triggers_per_min']:.3f}")
if sim['avg_latency_s'] is not None:
    print(f"Latency: {sim['avg_latency_s']:.2f}s +/- {sim['std_latency_s']:.2f}s")

## 8) Notes

- If false triggers are still too high, increase `enter_active_threshold`, `k_consecutive`, or `cooldown_windows`.
- If latency becomes too high, reduce `k_consecutive` or `cooldown_windows` slightly while tracking false trigger changes.
- Only after control logic is stable should feature/model upgrades be applied.